In [2]:
import os
import numpy as np
import pickle
import cv2
from mtcnn import MTCNN
from PIL import Image
from numpy import asarray
from numpy import expand_dims
from keras_facenet import FaceNet
from os import listdir

In [29]:
# Inisialisasi detektor MTCNN
detector = MTCNN()

# Inisialisasi FaceNet untuk ekstraksi fitur wajah
MyFaceNet = FaceNet()

# Folder yang berisi gambar wajah untuk pembuatan model
folder = 'data/'

# Database untuk menyimpan embedding wajah
database = {}

# Proses setiap gambar dalam folder untuk menghasilkan embeddings
for filename in os.listdir(folder):
    path = os.path.join(folder, filename)
    img = cv2.imread(path)

    # Deteksi wajah menggunakan MTCNN
    faces = detector.detect_faces(img)

    # Jika wajah terdeteksi, proses wajah tersebut
    if len(faces) > 0:
        for face in faces:
            # Ambil posisi dan ukuran wajah
            x1, y1, width, height = face['box']
            x2, y2 = x1 + width, y1 + height

            # Potong wajah dari gambar
            rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            cropped_face = rgb_img[y1:y2, x1:x2]

            # Validasi dimensi wajah
            if cropped_face.shape[0] == 0 or cropped_face.shape[1] == 0:
                print(f"Wajah pada {filename} tidak valid, dilewati.")
                continue

            # Resize wajah ke 160x160 (dimensi yang diterima FaceNet)
            face = Image.fromarray(cropped_face).resize((160, 160))
            face = asarray(face)

            # Ekspansi dimensi untuk prediksi model
            face = expand_dims(face, axis=0)
            signature = MyFaceNet.embeddings(face)

            # Ambil nama dan NIM dari filename (pastikan sesuai format file)
            name, nim_with_extension = filename.split('_')  # Example: "Ali_12345.jpg"
            nim = nim_with_extension.split('.')[0]  # Remove the '.jpg' extension

            # Simpan signature (embedding wajah) ke dalam database
            database[filename] = {
                'embedding': signature,
                'name': name,
                'nim': nim
            }
    else:
        # Jika tidak ada wajah yang terdeteksi, tetap lakukan embedding pada gambar secara keseluruhan
        print(f"Tidak ada wajah yang terdeteksi pada {filename}, tetap melakukan embedding pada gambar secara keseluruhan.")

        # Convert gambar ke RGB dan resize untuk proses embedding
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        resized_img = Image.fromarray(rgb_img).resize((160, 160))
        img_array = asarray(resized_img)

        # Ekspansi dimensi untuk prediksi model
        img_array = expand_dims(img_array, axis=0)
        signature = MyFaceNet.embeddings(img_array)

        # Ambil nama dan NIM dari filename (pastikan sesuai format file)
        name, nim_with_extension = filename.split('_')  # Example: "Ali_12345.jpg"
        nim = nim_with_extension.split('.')[0]  # Remove the '.jpg' extension

        # Simpan signature (embedding gambar secara keseluruhan) ke dalam database
        database[filename] = {
            'embedding': signature,
            'name': name,
            'nim': nim
        }

# Simpan database ke file .pkl untuk digunakan dalam aplikasi Streamlit
with open('model.pkl', 'wb') as f:
    pickle.dump(database, f)

print("Model telah dibuat dan disimpan ke model.pkl.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
Tidak ada wajah yang terdeteksi pada Unknown_.jpg, tetap melakukan embedding pada gambar secara keseluruhan.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
Model telah dibuat dan disimpan ke model.pkl.


In [30]:
database

{'Jeff Bezos_23SA11A700.jpg': {'embedding': array([[ 2.37701293e-02, -1.43024661e-02,  2.68743057e-02,
          -5.60753001e-03,  7.60025531e-02,  6.60225153e-02,
           1.25986105e-02, -3.98541997e-05, -7.20890164e-02,
           2.48080119e-02,  7.52403634e-04, -5.91862090e-02,
          -5.42985415e-03,  5.65253012e-02, -4.87890234e-03,
          -5.70128337e-02, -4.01367173e-02,  5.41744195e-02,
          -2.30293702e-02, -4.54682671e-02, -8.34089220e-02,
           4.07034419e-02,  7.03173876e-02,  2.54614968e-02,
           1.11080883e-02,  1.23354215e-02,  2.07572188e-02,
          -2.56184973e-02,  4.51306850e-02, -7.35567138e-02,
          -6.66962713e-02,  1.84525643e-02,  4.59985062e-02,
           9.56576434e-04,  8.89201090e-02,  2.84879487e-02,
           3.33809666e-02,  6.25639334e-02, -1.33322049e-02,
          -7.78486058e-02, -1.78222004e-02,  3.03475130e-02,
          -2.75322166e-03, -8.14105123e-02, -2.80923434e-02,
          -3.30260806e-02, -2.44392436e-02,

In [31]:
# Simpan database ke file .pkl
with open("model.pkl", "wb") as myfile:
    pickle.dump(database, myfile)

print("Model embeddings disimpan ke model.pkl")

Model embeddings disimpan ke model.pkl


In [32]:
# HaarCascade = cv2.CascadeClassifier(cv2.samples.findFile(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'))
#
# MyFaceNet = FaceNet()

In [33]:
# database = {}
# folder = 'data/'
#
# for filename in listdir(folder):
#     try:
#         if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
#             print(f"File {filename} bukan gambar, dilewati.")
#             continue
#
#         path = os.path.join(folder, filename)
#         img = cv2.imread(path)
#
#         wajah = HaarCascade.detectMultiScale(img, 1.1, 4)
#
#         if len(wajah)>0:
#             x1, y1, width, height = wajah[0]
#         else:
#             x1, y1, width, height = 1, 1, 10, 10
#
#         x1, y1 = abs(x1), abs(y1)
#         x2, y2 = x1 + width, y1 + height
#
#         # Konversi ke RGB dan potong wajah
#         gbr = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#         gbr = Image.fromarray(gbr)
#         gbr_array = asarray(gbr)
#
#         face = gbr_array[y1:y2, x1:x2]
#
#         # Resize wajah ke 160x160
#         face = Image.fromarray(face)
#         face = face.resize((160, 160))
#         face = asarray(face)
#
#         # Ekspansi dimensi untuk model
#         face = expand_dims(face, axis=0)
#
#         signature = MyFaceNet.embeddings(face)
#
#         database[os.path.splitext(filename)[0]] = signature
#         print(f"Berhasil memproses file {filename}.")
#
#     except Exception as e:
#         print(f"Gagal memproses file {filename}: {e}")